# <mark style="display:block; background:#d1c4e9; color:#1a1a1a; padding:6px 12px; border-radius:4px">10/6(화) 오후 · AI 에이전트 · 도구 호출 — 실습</mark>

오전에 `llm_client.py` 를 만들었습니다. 질문을 보내고 답을 JSON 으로 받아 읽을 수 있습니다.

오후에는 LLM 에게 **무엇을 할지 고르게** 합니다. 도구를 만들고 → 고르게 하고 → 우리 코드가 실행하고 → 위험한 일 앞에는 사람을 둡니다.

## <mark style="display:block; background:#c8e6c9; color:#1a1a1a; padding:6px 12px; border-radius:4px">시작하기</mark>

### 0.1 맨 먼저 · 이 파일을 놓는 자리

1. 단톡방에서 받은 이 파일을 `security-agent-toolkit` 폴더 안의 **`agent_core` 폴더**에 넣습니다. 오전 노트북과 같은 폴더입니다.
2. VS Code 에서 `security-agent-toolkit` 폴더를 열고, 왼쪽 목록의 `agent_core` 에서 이 파일을 엽니다.
3. 오른쪽 위 **커널 선택(Select Kernel)** 을 눌러 **각자 만든 가상환경**을 고릅니다. 오전과 같은 것입니다. 목록에서 이름에 `.venv` 처럼 가상환경 이름이 붙은 항목입니다. 그냥 `Python 3.x` 만 적힌 항목이 아닙니다.
4. 「시작하기」 맨 아래의 **위치 확인 셀**을 실행합니다. `준비 완료` 가 나와야 합니다.

### ⚠ 중요 · 무료 키는 1분에 15번까지만 호출할 수 있습니다

<div style="background:#fff3e0; color:#4e342e; padding:12px 16px; border-radius:8px; border-left:6px solid #e65100">
<b>무료 Gemini API 키의 한도는 1분에 15번입니다.</b><br>
1. 한도를 넘으면 상태 코드 <code>429</code> 가 반환되고 답을 받지 못합니다.<br>
2. <code>429</code> 가 나오면 <b>1분 기다린 뒤</b> 같은 셀을 다시 실행합니다. 키가 고장 난 것이 아닙니다.<br>
3. 같은 셀을 연달아 여러 번 실행하지 않습니다. LLM 을 세 번 호출하는 문제는 한 번 실행에 세 번을 씁니다.<br>
4. 한도에 걸리면 <code>[LLM 호출 실패] 상태 코드 429</code> 가 먼저 보이고, 그 아래에 <code>KeyError: 'choices'</code> 나 <code>TypeError: 'NoneType' object is not subscriptable</code> 같은 에러가 이어질 수 있습니다. <b>코드가 틀린 것이 아닙니다.</b> 1분 기다린 뒤 같은 셀을 다시 실행합니다.<br>
5. LLM 의 답 문장과 위험도는 실행할 때마다 달라질 수 있습니다. 🎯 와 글자가 같지 않아도 <b>모양이 같으면</b> 맞습니다.
</div>

### 0.2 오늘 오후의 순서

| 교시 | 무엇 |
|---|---|
| 5교시 | AI 에이전트와 도구 호출 — 도구를 만들고 LLM 이 고르게 한다 |
| 6교시 | 라우터 — 이름으로 함수를 찾아 실행한다 · `tool_router.py` |
| 7교시 | 승인 게이트 — 위험한 도구 앞에 사람을 둔다 · `agent_result.json` |
| 8교시 | 오늘 만든 것 확인 · 남은 도전 |

### 0.3 막혔을 때

1. 문제 앞에 **🐍 문법 상자**가 있으면 먼저 읽습니다. 그 문제에서 쓰는 문법입니다.
2. 문제 아래 **💡 힌트**를 순서대로 따라 합니다.
3. 그래도 막히면 노트북 맨 아래 「정답」으로 갑니다.
4. 도전 문제는 안 풀고 넘어가도 됩니다.

### 0.4 문제를 푸는 법

1. 코드 셀의 번호 주석이 **무엇을 쓸지** 알려 줍니다. 주석 바로 아래 빈 줄에 코드를 씁니다.
2. 주석에 적힌 **변수 이름을 그대로** 씁니다. 아래 `print` 줄이 그 이름을 쓰기 때문입니다.
3. 미리 채워 둔 줄은 앞에서 배운 것입니다. 고치지 않습니다.
4. 교시마다 되풀이되는 코드는 **준비 셀의 함수**로 줄여 씁니다. 5교시 `5.3` 에서 한 줄씩 설명합니다.
5. `NameError` 가 나오면 **그 교시의 준비 셀**을 실행하지 않은 것입니다. 준비 셀을 실행한 뒤 다시 합니다.

In [1]:
# 위치 확인 셀 — 그대로 실행합니다. 뒤의 셀이 이 셀의 ENV_PATH 를 씁니다
import os                                                     # 폴더 · 파일을 다루는 도구
import sys                                                    # 지금 실행 중인 파이썬의 정보를 주는 도구


def find_env():                                               # .env 를 지금 폴더부터 위쪽으로 찾는 함수
    folder = os.getcwd()                                      # 지금 폴더에서 시작한다
    for i in range(5):                          # 지금 폴더에서 다섯 칸 위까지 찾습니다
        path = os.path.join(folder, ".env")                   # 그 폴더 안의 .env 경로를 만든다
        if os.path.exists(path):                              # 그 파일이 있으면
            return path                                       # 찾은 경로를 돌려준다
        folder = os.path.dirname(folder)        # 한 칸 위 폴더
    return None


ENV_PATH = find_env()                                         # 찾은 .env 경로. 못 찾으면 None

print("노트북이 실행되는 폴더:", os.getcwd())
print("지금 쓰는 파이썬     :", sys.executable)

if ENV_PATH:
    print("키 파일(.env)         :", ENV_PATH)
    print("준비 완료")
else:
    print(".env 를 찾지 못했습니다")
    print("1) 이 노트북을 security-agent-toolkit 폴더 안의 agent_core 폴더로 옮긴 뒤 다시 엽니다")
    print("2) .env 가 security-agent-toolkit 폴더에 있는지 봅니다 (아침 과제 5)")

노트북이 실행되는 폴더: c:\Users\User\Documents\security-agent-toolkit\agent_core
지금 쓰는 파이썬     : c:\Users\User\Documents\security-agent-toolkit\.venv\Scripts\python.exe
키 파일(.env)         : c:\Users\User\Documents\security-agent-toolkit\agent_core\.env
준비 완료


---

# <mark style="display:block; background:#ffe0b2; color:#1a1a1a; padding:6px 12px; border-radius:4px">5교시 (14:00–14:50) · AI 에이전트와 도구 호출</mark>

아래 준비 셀 넷을 먼저 실행합니다. **오전을 못 끝냈어도 여기서부터 시작할 수 있습니다.**

1. `requests` 설치
2. 9/29 에 만든 로그 파일 `raw_logs.txt` 다시 만들기
3. `normalized_logs.json` 다시 만들기 — 오후의 도구가 읽는 파일입니다
4. 오전 산출물 `llm_client.py` 다시 만들기 — 오전에 완성했다면 같은 내용으로 덮어씁니다

In [2]:
%pip install -q requests

Note: you may need to restart the kernel to use updated packages.


In [3]:
%%writefile raw_logs.txt
2026-09-29 09:02:11 INFO accepted login for kim.cs from 10.1.2.11
2026-09-29 09:12:00 WARN failed login for kim01 from 203.0.113.5
2026-09-29 09:15:31 INFO accepted login for lee.yh from 10.1.2.34
2026-09-29 09:16:02 INFO session closed for 10.1.2.34
2026-09-29 10:03:19 WARN failed login for park.js from 10.1.3.7
2026-09-29 10:03:31 INFO accepted login for park.js from 10.1.3.7
2026-09-29 11:20:55 INFO accepted login for choi.mk from 10.1.4.2
2026-09-29 12:40:12 INFO session closed for 10.1.4.2
2026-09-29 03:11:05 WARN failed login for admin from 211.45.12.9
2026-09-29 03:12:47 WARN failed login for admin from 211.45.12.9
2026-09-29 03:13:58 WARN failed login for admin from 211.45.12.9
2026-09-29 03:15:22 WARN failed login for admin from 211.45.12.9
2026-09-29 03:17:09 INFO accepted login for admin from 211.45.12.9
2026-09-29 14:05:38 INFO accepted login for jung.hw from 10.1.2.88
2026-09-29 22:14:03 WARN failed login for kim.cs from 185.220.101.34
2026-09-29 22:14:21 WARN failed login for lee.yh from 185.220.101.34
2026-09-29 22:14:40 WARN failed login for choi.mk from 185.220.101.34
2026-09-29 22:14:58 WARN failed login for jung.hw from 185.220.101.34
2026-09-29 22:15:12 INFO session closed for 185.220.101.34
2026-09-29 23:40:07 WARN failed login for invalid user guest from 185.220.101.34

Writing raw_logs.txt


In [4]:
import re                                                     # 정규표현식 도구 (9/29)
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구

# 시각 · 등급 · 계정 · IP 를 꺼내는 정규표현식 (9/29)
PATTERN = r"(?P<time>\d{2}:\d{2}:\d{2}) (?P<level>\w+) \w+ login for (?P<user>[\w.]+) from (?P<ip>[\d.]+)"

rows = []
unmatched = []

with open("raw_logs.txt", encoding="utf-8") as f:
    for line in f:                                            # 파일을 한 줄씩 꺼낸다
        m = re.search(PATTERN, line)                          # 한 줄에서 패턴을 찾는다
        if m:
            rows.append(m.groupdict())                        # 찾은 칸들을 딕셔너리로 모은다
        else:
            unmatched.append(line.strip())                    # 패턴에 안 맞는 줄은 따로 모은다

with open("normalized_logs.json", "w", encoding="utf-8") as f:  # 정리한 결과를 파일로 쓴다
    json.dump(rows, f, ensure_ascii=False, indent=2)          # 한글 그대로, 두 칸 들여쓰기로 저장한다

print(f"정규화 {len(rows)}건 / 안 맞음 {len(unmatched)}건")

정규화 16건 / 안 맞음 4건


In [5]:
%%writefile llm_client.py
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구
import os                                                     # 폴더 · 파일을 다루는 도구
import requests                                               # 인터넷으로 요청을 보내는 도구 (9/30)

# Gemini 에 요청을 보낼 주소
URL = "https://generativelanguage.googleapis.com/v1beta/openai/chat/completions"
MODEL = "gemini-3.5-flash-lite"                               # 쓸 모델 이름


def find_env():                                               # .env 를 지금 폴더부터 위쪽으로 찾는 함수
    folder = os.getcwd()                                      # 지금 폴더에서 시작한다
    for i in range(5):                          # 지금 폴더에서 다섯 칸 위까지 찾습니다
        path = os.path.join(folder, ".env")                   # 그 폴더 안의 .env 경로를 만든다
        if os.path.exists(path):                              # 그 파일이 있으면
            return path                                       # 찾은 경로를 돌려준다
        folder = os.path.dirname(folder)        # 한 칸 위 폴더
    return None


def read_api_key():                                           # .env 에서 키를 읽어 돌려주는 함수
    api_key = None                                            # 키를 못 찾으면 None 으로 남는다
    with open(find_env(), encoding="utf-8") as f:             # .env 를 찾아 읽기로 연다
        for line in f:                                        # 파일을 한 줄씩 꺼낸다
            parts = line.strip().split("=", 1)                # 줄 끝 공백을 지우고 = 에서 둘로 나눈다
            if parts[0] == "GEMINI_API_KEY":                  # = 앞부분이 키 이름이면
                api_key = parts[1]                            # = 뒷부분이 키 값이다
    return api_key                                            # 읽은 키를 돌려준다


def call_llm(question):                                       # 질문을 받아 답 문장을 돌려주는 함수
    # 요청 헤더 — .env 에서 읽은 키를 넣는다
    headers = {"Authorization": "Bearer " + read_api_key(), "Content-Type": "application/json"}
    # 요청 본문 — 모델 · temperature 0 · 질문
    body = {"model": MODEL, "temperature": 0, "messages": [{"role": "user", "content": question}]}
    # POST 요청을 보낸다. 30초 안에 답이 없으면 멈춘다
    response = requests.post(URL, headers=headers, json=body, timeout=30)
    if response.status_code != 200:                           # 200(성공)이 아니면
        print("[LLM 호출 실패] 상태 코드", response.status_code)      # 429 면 1분 기다린 뒤 다시 실행
        return ""                                             # 실패하면 빈 문자열을 돌려준다
    data = response.json()                                    # 응답(JSON 문자열)을 딕셔너리로 바꾼다
    return data["choices"][0]["message"]["content"]           # 답 문장만 돌려준다


def parse_llm_json(text):                                     # LLM 의 답을 딕셔너리로 바꾸는 함수. 못 읽으면 None
    clean = text.replace("```json", "")                       # 코드 블록 표시 ```json 을 지운다
    clean = clean.replace("```", "")                          # 남은 ``` 를 지운다
    clean = clean.strip()                                     # 앞뒤 공백 · 줄바꿈을 지운다
    try:                                                      # 아래 줄을 해 본다
        return json.loads(clean)                              # 문자열을 딕셔너리로 바꿔 돌려준다
    except json.JSONDecodeError:                              # JSON 으로 읽지 못하면 여기로 온다
        return None

Overwriting llm_client.py


## <mark style="display:block; background:#b3e5fc; color:#0d3c61; padding:6px 12px; border-radius:4px">🔎 공부하는 법을 공부하기</mark>

설명을 읽기 전에 아래 **두 개념**을 스스로 찾아봅니다.

| 찾아볼 개념 | 알아 올 것 |
|---|---|
| **AI 에이전트 (agent)** | 묻고 답하는 챗봇과 무엇이 다른가 |
| **도구 호출 (tool use · function calling)** | LLM 이 우리 컴퓨터의 함수를 직접 실행하는가 |

**여기에 기록하세요** — 이 셀을 **두 번 눌러** 아래에 적습니다.

- AI 에이전트 →
- 도구 호출 →

---

## <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">5 · 도구를 만들고 LLM 이 고르게 한다</mark>

### 왜 필요한가

1. 오전의 LLM 은 로그를 **설명**하기만 했습니다. 로그 파일을 세거나 IP 를 조회하지는 못합니다.
2. 2교시에 봤듯이 LLM 은 조회하지 않고 지어냅니다. **사실은 우리 코드가 확인**해야 합니다.
3. 그래서 일을 나눕니다. 우리가 **도구(함수)** 를 만들어 두고, LLM 은 **어느 도구를 쓸지 고르기만** 합니다. 실행은 우리 코드가 합니다.

### 5교시에 사용하는 용어

| 용어 (읽는 법) | 뜻 | 5교시의 예 |
|---|---|---|
| 에이전트 (agent) | 목표를 받아 스스로 순서를 정하고 도구를 쓰는 프로그램 | 목표 → 도구 고르기 → 실행 → 답 |
| 도구 (tool) | LLM 이 고를 수 있게 우리가 만들어 둔 함수 | `count_failed_logins(args)` |
| 도구 호출 (tool calling) | 모델이 어떤 함수를 어떤 값으로 부를지 정하는 구조 | `{"tool": "count_failed_logins", "args": {"user": "admin"}}` |
| 도구 목록 | 쓸 수 있는 함수의 이름·설명·인자를 적은 글 | `tools_text` |

### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">5.1 도구는 딕셔너리 하나를 받는 함수입니다</mark>

도구는 평범한 파이썬 함수입니다. 다만 인자를 **딕셔너리 하나(`args`)** 로 받게 만듭니다. LLM 이 인자를 JSON 으로 주기 때문입니다.

```python
import json

def count_failed_logins(args):
    with open("normalized_logs.json", encoding="utf-8") as f:
        rows = json.load(f)
    count = 0
    for row in rows:
        if row["user"] == args["user"] and row["level"] == "WARN":
            count = count + 1
    return count

print(count_failed_logins({"user": "kim.cs"}))          # 1
```

1. `args["user"]` 로 어느 계정을 셀지 받습니다.
2. 9/29 오전의 룰 ① 과 같은 코드입니다. 계정이 같고 등급이 `WARN` 인 기록을 셉니다.
3. 결과는 `return` 으로 반환합니다. 화면에 출력하지 않습니다. 부른 쪽이 그 값을 다시 씁니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 5-1 · 무엇이 보일까요</font></h3></td></tr></table>

아래 코드를 실행하면 화면에 무엇이 보일지 적어 보세요. 적은 뒤에 셀을 실행해 맞춰 봅니다.

```python
def count_warn(args):
    rows = [{"user": "jdoe", "level": "WARN"},
            {"user": "jdoe", "level": "INFO"},
            {"user": "ops.kim", "level": "WARN"}]
    count = 0
    for row in rows:
        if row["user"] == args["user"] and row["level"] == "WARN":
            count = count + 1
    return count

print(count_warn({"user": "jdoe"}))
```

막히면 바로 위 `5.1 도구는 딕셔너리 하나를 받는 함수입니다` 설명을 다시 봅니다.

In [6]:
def count_warn(args):
    rows = [{"user": "jdoe", "level": "WARN"},
            {"user": "jdoe", "level": "INFO"},
            {"user": "ops.kim", "level": "WARN"}]
    count = 0
    for row in rows:
        if row["user"] == args["user"] and row["level"] == "WARN":
            count = count + 1
    return count

print(count_warn({"user": "jdoe"}))

1


✅ `1` — `jdoe` 의 기록은 둘이지만 `WARN` 은 하나입니다.

#### 🐍 문법 상자 · `and` — 두 조건이 모두 참일 때만

```python
rows = [{"user": "kim", "level": "WARN"},
        {"user": "kim", "level": "INFO"},
        {"user": "lee", "level": "WARN"}]
count = 0
for row in rows:
    if row["user"] == "kim" and row["level"] == "WARN":
        count = count + 1
print(count)
# 1
```

| 기록 | 계정이 kim | 등급이 WARN | `and` |
|---|---|---|---|
| kim · WARN | 참 | 참 | **참** — 센다 |
| kim · INFO | 참 | 거짓 | 거짓 |
| lee · WARN | 거짓 | 참 | 거짓 |

⚠ `return count` 를 `for` 안에 넣으면 첫 기록을 보자마자 함수가 끝납니다. `for` 와 같은 깊이에 씁니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 5-2 · 로그인 실패를 세는 도구 만들기</font></h3></td></tr></table>

계정 하나의 로그인 실패 횟수를 반환하는 도구 `count_failed_logins` 를 완성하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `4` |

**💡 힌트**

1. 조건은 둘입니다 — 계정이 `args["user"]` 와 같고, 등급이 `WARN` 입니다. `and` 로 잇습니다.
2. 조건에 맞으면 `count` 에 1 을 더합니다.
3. 반복이 끝난 뒤에 `count` 를 반환합니다. `return` 은 `for` 와 같은 깊이에 씁니다.

In [7]:
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구


def count_failed_logins(args):                                # 도구 — 한 계정의 로그인 실패 횟수를 센다
    with open("normalized_logs.json", encoding="utf-8") as f:  # 9/29 에 정리한 로그 파일을 연다
        rows = json.load(f)                                   # 파일 내용을 리스트로 읽는다
    count = 0                                                 # 센 횟수. 0 에서 시작
    for row in rows:                                          # 로그를 한 건씩 꺼낸다
        # 1. row 의 user 가 args 의 user 와 같고 level 이 "WARN" 이면 (if 문)
        if row["user"] == args["user"] and row["level"] == "WARN":
            # 2. count 에 1 을 더하세요
            count = count + 1
    # 3. count 를 return 하세요
    return count


print(count_failed_logins({"user": "admin"}))

4


### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">5.2 도구 목록을 보내면 LLM 이 하나를 고릅니다</mark>

프롬프트에 **도구 목록과 목표**를 함께 넣습니다. LLM 은 어느 도구를 어떤 값으로 부를지를 JSON 으로 답합니다.

```python
tools_text = "쓸 수 있는 도구는 둘입니다. "
tools_text = tools_text + "1) count_failed_logins: 한 계정의 로그인 실패 횟수를 센다. 인자는 user. "
tools_text = tools_text + "2) lookup_ip: IP 주소의 나라와 통신사를 조회한다. 인자는 ip. "
tools_text = tools_text + "목표를 이루는 데 쓸 도구 하나를 골라 JSON 으로만 답하세요. "
tools_text = tools_text + '예시: {"tool": "count_failed_logins", "args": {"user": "kim01"}} '

text = call_llm(tools_text + "목표: admin 계정이 로그인에 몇 번 실패했는지 알려 주세요.")
```

실제로 받은 답입니다.

```
```json
{"tool": "count_failed_logins", "args": {"user": "admin"}}
```
```

1. LLM 은 **고르기만** 했습니다. 로그 파일을 열지도, 함수를 실행하지도 않았습니다.
2. 고른 결과는 문자열입니다. 오전에 만든 `parse_llm_json` 으로 딕셔너리로 바꿉니다.
3. **실행은 우리 파이썬 코드가 합니다.** 그래서 실행할지 말지를 우리가 정할 수 있습니다.

#### 🐍 문법 상자 · 딕셔너리 안의 딕셔너리

```python
choice = {"tool": "count_failed_logins", "args": {"user": "kim.cs"}}
print(choice["tool"])
print(choice["args"]["user"])
# count_failed_logins
# kim.cs
```

| 쓰는 것 | 결과 |
|---|---|
| `choice["tool"]` | 고른 도구 이름 — 문자열 |
| `choice["args"]` | 인자 — 딕셔너리 |
| `choice["args"]["user"]` | 그 안의 계정 이름 |

⚠ LLM 이 고른 것은 아직 **이름**일 뿐입니다. 함수는 우리 코드가 실행합니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 5-3 · LLM 에게 도구를 고르게 하고, 고른 것을 읽기</font></h3></td></tr></table>

도구 목록과 목표를 보내 LLM 의 답을 **문자열 그대로** 출력하고, 그 답을 딕셔너리로 바꿔 **도구 이름**과 **인자**를 한 줄씩 출력하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `{"tool": "count_failed_logins", "args": {"user": "kim.cs"}}` 가 든 문자열(코드 블록 표시가 붙을 때도 있습니다) · `count_failed_logins` · `{'user': 'kim.cs'}` |

**💡 힌트**

1. 질문 문자열은 `tools_text`, `"목표: "`, `goal` 을 순서대로 이어 붙입니다.
2. `call_llm` 에 질문 문자열을 넣으면 응답 문자열이 반환됩니다. 아직 아무 도구도 실행되지 않았습니다.
3. 문자열을 딕셔너리로 바꾸는 것은 오전에 만든 `parse_llm_json` 입니다. 도구 이름은 `tool` 키에, 인자는 `args` 키에 있습니다.

In [8]:
import importlib                                              # 파일을 다시 읽는 도구
import llm_client                                             # 오전에 만든 llm_client.py 를 불러온다

importlib.reload(llm_client)          # llm_client.py 를 고쳤으면 다시 읽습니다
from llm_client import call_llm, parse_llm_json               # llm_client.py 의 두 함수를 꺼내 쓴다

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
tools_text = "쓸 수 있는 도구는 둘입니다. "                              # LLM 에게 알려 줄 도구 목록을 한 줄씩 이어 붙인다
tools_text = tools_text + "1) count_failed_logins: 한 계정의 로그인 실패 횟수를 센다. 인자는 user. "
tools_text = tools_text + "2) lookup_ip: IP 주소의 나라와 통신사를 조회한다. 인자는 ip. "
tools_text = tools_text + "목표를 이루는 데 쓸 도구 하나를 골라 JSON 으로만 답하세요. "
tools_text = tools_text + '예시: {"tool": "count_failed_logins", "args": {"user": "kim01"}} '
goal = "kim.cs 계정이 로그인에 몇 번 실패했는지 알려 주세요."
# ── 여기까지 ──

# 1. prompt 라는 변수를 만들어 tools_text, "목표: ", goal 을 이어 붙이세요
prompt = tools_text + "목표: " + goal

# 2. text 라는 변수를 만들어 call_llm 에 prompt 를 넣어 받은 답을 담고, text 를 출력하세요
text = call_llm(prompt)                                       # LLM 에 보내 답 문장을 받는다
print(text)

# 3. choice 라는 변수를 만들어 parse_llm_json 에 text 를 넣어 받은 것을 담으세요
choice = parse_llm_json(text)

# 4. choice 의 tool 키와 args 키를 한 줄씩 출력하세요
print(choice["tool"])
print(choice["args"])


{"tool": "count_failed_logins", "args": {"user": "kim.cs"}}
count_failed_logins
{'user': 'kim.cs'}


### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">5.3 되풀이되는 줄을 함수 `choose_tool` 로 묶습니다</mark>

문제 5-3 에서 쓴 **「질문을 만들어 보내고 → 답을 딕셔너리로 바꾸는」** 줄은 오후 내내 되풀이됩니다. 매번 다시 쓰지 않도록 함수 하나로 묶습니다.

```python
def choose_tool(goal):
    text = call_llm(tools_text + "목표: " + goal)
    if text == "":
        print("LLM 의 답을 받지 못했습니다. 1분 기다린 뒤 이 셀을 다시 실행합니다 (한도를 넘으면 429 가 옵니다)")
    return parse_llm_json(text)
```

| 줄 | 하는 일 | 문제 5-3 에서 쓴 줄 |
|---|---|---|
| `def choose_tool(goal):` | 목표 하나를 받는 함수를 만든다 | — |
| `text = call_llm(tools_text + "목표: " + goal)` | 도구 목록과 목표를 이어 붙여 LLM 에 보낸다 | 1번 · 2번 |
| `if text == "":` | 답을 받지 못했으면 안내를 출력한다 | — |
| `return parse_llm_json(text)` | 답을 딕셔너리로 바꿔 반환한다 | 3번 |

1. 쓰는 법은 `choice = choose_tool(goal)` 한 줄입니다. 문제 5-3 의 네 줄이 한 줄이 됩니다.
2. 반환값은 `{"tool": "…", "args": {…}}` 모양의 딕셔너리입니다. LLM 의 답을 읽지 못하면 `None` 입니다.
3. `call_llm` 은 한도를 넘으면 빈 문자열 `""` 을 반환합니다. 그때 안내가 출력됩니다.

아래 **준비 셀**을 실행합니다. `choose_tool` 과 함께, 도구 둘도 여기에 있습니다.

- `count_failed_logins` — 문제 5-2 에서 만든 것과 같은 함수입니다.
- `lookup_ip` — IP 주소의 나라와 통신사를 조회해 문자열로 반환하는 도구입니다. 오전 2교시에 쓴 조회 API(`ipwho.is`)를 씁니다.

In [9]:
# 준비 셀 — 그대로 실행합니다. 5교시의 나머지 문제가 이 셀의 함수를 씁니다
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구
import requests                                               # 인터넷으로 요청을 보내는 도구 (9/30)
import importlib                                              # 파일을 다시 읽는 도구
import llm_client                                             # 오전에 만든 llm_client.py 를 불러온다

importlib.reload(llm_client)          # llm_client.py 를 고쳤으면 다시 읽습니다
from llm_client import call_llm, parse_llm_json               # llm_client.py 의 두 함수를 꺼내 쓴다

tools_text = "쓸 수 있는 도구는 둘입니다. "                              # LLM 에게 알려 줄 도구 목록을 한 줄씩 이어 붙인다
tools_text = tools_text + "1) count_failed_logins: 한 계정의 로그인 실패 횟수를 센다. 인자는 user. "
tools_text = tools_text + "2) lookup_ip: IP 주소의 나라와 통신사를 조회한다. 인자는 ip. "
tools_text = tools_text + "목표를 이루는 데 쓸 도구 하나를 골라 JSON 으로만 답하세요. "
tools_text = tools_text + '예시: {"tool": "count_failed_logins", "args": {"user": "kim01"}} '


def choose_tool(goal):                                        # 목표를 받아 LLM 이 고른 도구를 딕셔너리로 돌려주는 함수 (5.3)
    text = call_llm(tools_text + "목표: " + goal)               # 도구 목록과 목표를 이어 LLM 에 보낸다
    if text == "":                                            # 답을 받지 못했으면 (한도 초과 등)
        print("LLM 의 답을 받지 못했습니다. 1분 기다린 뒤 이 셀을 다시 실행합니다 (한도를 넘으면 429 가 옵니다)")
    return parse_llm_json(text)                               # 답을 딕셔너리로 바꿔 돌려준다


def count_failed_logins(args):                                # 도구 — 한 계정의 로그인 실패 횟수를 센다
    with open("normalized_logs.json", encoding="utf-8") as f:  # 9/29 에 정리한 로그 파일을 연다
        rows = json.load(f)                                   # 파일 내용을 리스트로 읽는다
    count = 0                                                 # 센 횟수. 0 에서 시작
    for row in rows:                                          # 로그를 한 건씩 꺼낸다
        # 계정이 같고 등급이 WARN(실패)이면
        if row["user"] == args["user"] and row["level"] == "WARN":
            count = count + 1                                 # 하나 센다
    return count                                              # 센 횟수를 돌려준다


def lookup_ip(args):                                          # 도구 — IP 의 나라와 통신사를 조회한다
    # 조회 API 에 IP 를 물어 딕셔너리로 받는다
    data = requests.get("https://ipwho.is/" + args["ip"], timeout=10).json()
    return data["country"] + " / " + data["connection"]["isp"]  # 나라와 통신사를 " / " 로 이어 돌려준다


print("준비 끝 — tools_text · choose_tool · count_failed_logins · lookup_ip")

준비 끝 — tools_text · choose_tool · count_failed_logins · lookup_ip


### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">5.4 목표가 바뀌면 다른 도구를 고릅니다</mark>

같은 도구 목록에 목표만 바꿔 보냈을 때 실제로 받은 답입니다.

| 목표 | LLM 이 고른 것 |
|---|---|
| admin 계정이 로그인에 몇 번 실패했는지 알려 주세요. | `count_failed_logins` · `{"user": "admin"}` |
| 185.220.101.34 는 어느 나라의 주소인지 알려 주세요. | `lookup_ip` · `{"ip": "185.220.101.34"}` |
| admin 계정을 잠가 주세요. | `count_failed_logins` · `{"user": "admin"}` |
| 오늘 점심 메뉴를 추천해 주세요. | 「관련된 도구가 없습니다」 라는 문장, 또는 `{"tool": null, "args": {}}` |

1. 목표에 맞는 도구가 목록에 있으면 그것을 고릅니다.
2. **목록에 없는 일**(계정 잠금)을 시켰더니, 있는 도구 가운데 하나를 억지로 골랐습니다. 목표는 이뤄지지 않습니다.
3. 도구와 상관없는 목표에는 JSON 이 아닌 문장이 올 때도 있습니다. `parse_llm_json` 이 `None` 을 반환하는 경우입니다.
4. 그래서 LLM 이 고른 것을 **그대로 실행하지 않고 확인**합니다. 6교시와 7교시가 그 확인입니다.

#### 🐍 문법 상자 · `if` · `elif` — 갈래 고르기

```python
tool = "lookup_ip"
if tool == "count_failed_logins":
    print("세기")
elif tool == "lookup_ip":
    print("조회")
# 조회
```

| 쓰는 것 | 뜻 |
|---|---|
| `if 조건:` | 첫 갈래 |
| `elif 조건:` | 위가 거짓일 때 다음 갈래 |

⚠ 두 조건이 다 거짓이면 아무 줄도 실행되지 않습니다. 그 뒤에 `result` 를 쓰면 `NameError` 입니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 5-4 · 두 도구 가운데 고른 것을 실행하기</font></h3></td></tr></table>

LLM 이 고른 도구가 `count_failed_logins` 이면 그 함수를, `lookup_ip` 이면 그 함수를 실행해 결과를 출력하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `실행 결과: South Korea / SamsungSDS Inc.` |

**💡 힌트**

1. 갈래가 둘이므로 `if` 와 `elif` 를 씁니다.
2. 두 갈래 모두 `result` 라는 변수에 함수의 결과를 담습니다.
3. 2교시에 LLM 은 이 IP 의 통신사를 지어냈습니다. 이번에는 조회 도구가 실제 값을 가져옵니다.

In [10]:
# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
goal = "211.45.12.9 는 어느 나라의 어느 통신사 주소인지 알려 주세요."
choice = choose_tool(goal)                                    # LLM 이 고른 도구 — {"tool": 이름, "args": 인자}
# ── 여기까지 ──

# 1. choice 의 tool 이 "count_failed_logins" 이면 (if 문)
if choice["tool"] == "count_failed_logins":
    # 2. result 라는 변수에 count_failed_logins 에 choice 의 args 를 넣어 받은 값을 담으세요
    result = count_failed_logins(choice["args"])
# 3. choice 의 tool 이 "lookup_ip" 이면 (elif 문)
elif choice["tool"] == "lookup_ip":
    # 4. result 에 lookup_ip 에 choice 의 args 를 넣어 받은 값을 담으세요
    result = lookup_ip(choice["args"])

print("실행 결과:", result)

실행 결과: South Korea / SamsungSDS Inc.


### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">5.5 실행 결과를 다시 LLM 에 넘겨 답을 만듭니다</mark>

도구가 반환한 값은 숫자나 짧은 문자열입니다. 그 값을 다시 LLM 에 넘겨 **사람이 읽을 문장**으로 만듭니다.

```python
prompt2 = "목표: admin 계정이 로그인에 몇 번 실패했는지 알려 주세요. "
prompt2 = prompt2 + "도구 count_failed_logins 를 실행한 결과는 4 입니다. "
prompt2 = prompt2 + "이 결과로 목표에 한 문장으로 답하세요."

print(call_llm(prompt2))          # admin 계정의 로그인 실패 횟수는 총 4번입니다.
```

1. 목표 → 도구 고르기 → 실행 → 결과로 답하기. 이 한 바퀴가 에이전트의 기본 흐름입니다.
2. 숫자 `4` 는 LLM 이 지어낸 것이 아니라 **우리 도구가 로그 파일에서 센 값**입니다.
3. 숫자를 문자열에 이어 붙일 때는 9/22 에 배운 `str()` 로 문자열로 바꿉니다.

#### 🐍 문법 상자 · 숫자를 문자열에 붙일 때는 `str()`

```python
result = 4
print("결과는 " + str(result) + "번")
# 결과는 4번
```

| 쓰는 것 | 결과 |
|---|---|
| `"결과는 " + str(result)` | `"결과는 4"` |
| `"결과는 " + result` | `TypeError: can only concatenate str (not "int") to str` |

⚠ `print("결과는", result)` 처럼 쉼표로 넘기면 `str()` 이 없어도 됩니다. 이어 붙이는 `+` 에서만 필요합니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 5-5 · 실행 결과로 최종 답 만들기</font></h3></td></tr></table>

도구를 실행해 얻은 `result` 를 LLM 에 다시 넘겨, 목표에 대한 **한 문장 답**을 출력하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `kim.cs 계정의 로그인 실패 횟수는 1번입니다.` 와 비슷한 한 문장 |

**💡 힌트**

1. 두 번째 질문 문자열에는 목표, 실행 결과, 「한 문장으로 답하세요」 를 넣습니다.
2. `result` 는 숫자입니다. 문자열에 이어 붙이려면 `str(result)` 로 바꿉니다.
3. 이번 호출은 도구를 고르는 것이 아니라 문장을 만드는 것입니다. `choose_tool` 이 아니라 `call_llm` 을 씁니다.

In [11]:
# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
goal = "kim.cs 계정이 로그인에 몇 번 실패했는지 알려 주세요."
result = count_failed_logins({"user": "kim.cs"})      # LLM 이 고른 도구를 실행했다고 가정합니다
# ── 여기까지 ──

# 1. prompt2 라는 변수를 만들어 "목표: " + goal 뒤에 " 도구를 실행한 결과는 " 과 str(result) 를 이어 붙이세요
prompt2 = "목표: " + goal + " 도구를 실행한 결과는 " + str(result)

# 2. prompt2 뒤에 " 입니다. 이 결과로 목표에 한 문장으로 답하세요." 를 이어 붙여 다시 prompt2 에 담으세요
prompt2 = prompt2 + " 입니다. 이 결과로 목표에 한 문장으로 답하세요."

# 3. final 이라는 변수를 만들어 call_llm 에 prompt2 를 넣어 받은 답을 담으세요
final = call_llm(prompt2)

print(final)

kim.cs 계정의 로그인 실패 횟수는 1번입니다.


<div style="background:#fff3e0; color:#4e342e; padding:12px 16px; border-radius:8px"><strong>⭐ 도전 문제 (선택)</strong><br>기본 문제를 다 푼 사람만 풉니다. 못 풀어도 괜찮습니다.</div>

<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 5-6 · 목표 두 개를 차례로 처리하기</font></h3></td></tr></table>

`goals` 의 목표 둘을 차례로 처리하시오. 목표마다 LLM 이 도구를 고르고, 우리 코드가 실행하고, `고른 도구 → 결과` 형식으로 출력합니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `count_failed_logins → 4` 그리고 `lookup_ip → Germany / Stiftung Erneuerbare Freiheit` |

**💡 힌트**

1. 반복 안에서 고르기, `if`·`elif` 로 실행하기를 차례로 합니다. 문제 5-4 와 같은 방법입니다.
2. 고르는 줄은 `choice = choose_tool(goal)` 입니다.
3. 출력은 `print(choice["tool"], "→", result)` 로 합니다.

In [12]:
# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
goals = [
    "admin 계정이 로그인에 몇 번 실패했는지 알려 주세요.",
    "185.220.101.34 는 어느 나라의 주소인지 알려 주세요.",
]
# ── 여기까지 ──

# 1. goals 를 하나씩 꺼내 반복하세요. 꺼낸 목표의 변수 이름은 goal 입니다
# 2. 반복 안에서 choice 에 choose_tool(goal) 의 결과를 담으세요
# 3. choice 의 tool 에 따라 if · elif 로 알맞은 도구를 실행해 result 에 담으세요
# 4. choice 의 tool, "→", result 를 출력하세요


### 📋 5교시 한눈에

| 하려는 일 | 쓰는 코드 |
|---|---|
| 도구를 만든다 | `def count_failed_logins(args):` … `return count` |
| LLM 에게 고르게 한다 | `call_llm(tools_text + "목표: " + goal)` |
| 고른 것을 읽는다 | `choice = parse_llm_json(text)` · `choice["tool"]` · `choice["args"]` |
| 위 두 줄을 한 번에 | `choice = choose_tool(goal)` |
| 우리 코드로 실행한다 | `if choice["tool"] == "…":` `result = 도구(choice["args"])` |
| 결과로 답을 만든다 | `call_llm("목표: … 실행한 결과는 … 한 문장으로 답하세요.")` |

LLM 은 고르기만 합니다. 실행은 우리 코드가 합니다.

---

# <mark style="display:block; background:#ffe0b2; color:#1a1a1a; padding:6px 12px; border-radius:4px">6교시 (15:00–15:50) · 라우터 — 이름으로 함수를 찾아 실행한다</mark>

## <mark style="display:block; background:#b3e5fc; color:#0d3c61; padding:6px 12px; border-radius:4px">🔎 공부하는 법을 공부하기</mark>

설명을 읽기 전에 아래 **두 개념**을 스스로 찾아봅니다.

| 찾아볼 개념 | 알아 올 것 |
|---|---|
| **라우터 (router)** | 프로그램에서 「라우팅」 은 무엇을 무엇으로 이어 주는 일인가 |
| **레지스트리 (registry)** | 이름과 실제 대상을 짝지어 두면 무엇이 편해지는가 |

**여기에 기록하세요** — 이 셀을 **두 번 눌러** 아래에 적습니다.

- 라우터 →
- 레지스트리 →

---

## <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">6 · 이름으로 함수를 찾아 실행한다</mark>

### 왜 필요한가

1. 5교시에는 도구마다 `if` · `elif` 를 한 갈래씩 썼습니다. 도구가 열 개면 갈래도 열 개입니다.
2. LLM 이 **목록에 없는 이름**을 답하면 어느 갈래에도 걸리지 않고 조용히 지나갑니다. 왜 아무 일도 안 일어났는지 아무도 모릅니다.
3. 그래서 **이름과 함수를 짝지은 딕셔너리**를 만들고, 이름으로 함수를 찾아 실행하는 코드를 한 곳에 둡니다. 없는 이름이면 그 자리에서 알립니다.

### 6교시에 사용하는 용어

| 용어 (읽는 법) | 뜻 | 6교시의 예 |
|---|---|---|
| 레지스트리 (registry) | 이름과 함수를 짝지어 둔 딕셔너리 | `tool_registry = {"lookup_ip": lookup_ip}` |
| 라우터 (router) | 이름을 보고 실제 함수를 찾아 실행하는 코드 | `route_tool_call(choice)` |

아래 **준비 셀**을 먼저 실행합니다. 5교시의 도구 둘과 `choose_tool` 을 다시 준비합니다. **5교시를 못 끝냈어도 여기서부터 시작할 수 있습니다.**

In [13]:
# 준비 셀 — 그대로 실행합니다. 6교시의 문제가 이 셀의 함수를 씁니다
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구
import requests                                               # 인터넷으로 요청을 보내는 도구 (9/30)
import importlib                                              # 파일을 다시 읽는 도구
import llm_client                                             # 오전에 만든 llm_client.py 를 불러온다

importlib.reload(llm_client)          # llm_client.py 를 고쳤으면 다시 읽습니다
from llm_client import call_llm, parse_llm_json               # llm_client.py 의 두 함수를 꺼내 쓴다

tools_text = "쓸 수 있는 도구는 둘입니다. "                              # LLM 에게 알려 줄 도구 목록을 한 줄씩 이어 붙인다
tools_text = tools_text + "1) count_failed_logins: 한 계정의 로그인 실패 횟수를 센다. 인자는 user. "
tools_text = tools_text + "2) lookup_ip: IP 주소의 나라와 통신사를 조회한다. 인자는 ip. "
tools_text = tools_text + "목표를 이루는 데 쓸 도구 하나를 골라 JSON 으로만 답하세요. "
tools_text = tools_text + '예시: {"tool": "count_failed_logins", "args": {"user": "kim01"}} '


def choose_tool(goal):                                        # 목표를 받아 LLM 이 고른 도구를 딕셔너리로 돌려주는 함수 (5.3)
    text = call_llm(tools_text + "목표: " + goal)               # 도구 목록과 목표를 이어 LLM 에 보낸다
    if text == "":                                            # 답을 받지 못했으면 (한도 초과 등)
        print("LLM 의 답을 받지 못했습니다. 1분 기다린 뒤 이 셀을 다시 실행합니다 (한도를 넘으면 429 가 옵니다)")
    return parse_llm_json(text)                               # 답을 딕셔너리로 바꿔 돌려준다


def count_failed_logins(args):                                # 도구 — 한 계정의 로그인 실패 횟수를 센다
    with open("normalized_logs.json", encoding="utf-8") as f:  # 9/29 에 정리한 로그 파일을 연다
        rows = json.load(f)                                   # 파일 내용을 리스트로 읽는다
    count = 0                                                 # 센 횟수. 0 에서 시작
    for row in rows:                                          # 로그를 한 건씩 꺼낸다
        # 계정이 같고 등급이 WARN(실패)이면
        if row["user"] == args["user"] and row["level"] == "WARN":
            count = count + 1                                 # 하나 센다
    return count                                              # 센 횟수를 돌려준다


def lookup_ip(args):                                          # 도구 — IP 의 나라와 통신사를 조회한다
    # 조회 API 에 IP 를 물어 딕셔너리로 받는다
    data = requests.get("https://ipwho.is/" + args["ip"], timeout=10).json()
    return data["country"] + " / " + data["connection"]["isp"]  # 나라와 통신사를 " / " 로 이어 돌려준다


print("준비 끝 — tools_text · choose_tool · count_failed_logins · lookup_ip")

준비 끝 — tools_text · choose_tool · count_failed_logins · lookup_ip


### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">6.1 함수도 딕셔너리의 값으로 담을 수 있습니다</mark>

함수 이름을 **괄호 없이** 쓰면 실행하지 않고 함수 자체를 가리킵니다. 그래서 딕셔너리의 값으로 담을 수 있습니다.

```python
def say_ok(args):
    return "ok " + args["name"]

registry = {"ok": say_ok}          # 괄호가 없다 — 실행하지 않고 담기만 한다

func = registry.get("ok")           # 이름으로 함수를 꺼낸다
print(func({"name": "deploy"}))     # 꺼낸 함수를 실행한다 → ok deploy

print(registry.get("없는이름"))      # None
```

1. `say_ok` 는 함수 자체이고, `say_ok(…)` 는 함수를 실행한 결과입니다. 10/2 오후 스케줄러에 함수를 넘길 때도 괄호 없이 썼습니다.
2. `.get()` 은 9/22 에 배웠습니다. 키가 없으면 오류를 내지 않고 `None` 을 반환합니다.
3. 도구가 늘어도 **딕셔너리에 한 줄만 더하면** 됩니다. 실행하는 코드는 바뀌지 않습니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 6-1 · 무엇이 보일까요</font></h3></td></tr></table>

아래 코드를 실행하면 화면에 무엇이 보일지 적어 보세요. 적은 뒤에 셀을 실행해 맞춰 봅니다.

```python
def double(args):
    return args["n"] * 2

registry = {"double": double}
func = registry.get("double")

print(func({"n": 21}))
```

막히면 바로 위 `6.1 함수도 딕셔너리의 값으로 담을 수 있습니다` 설명을 다시 봅니다.

In [14]:
def double(args):
    return args["n"] * 2

registry = {"double": double}
func = registry.get("double")

print(func({"n": 21}))

42


✅ `42` — 이름 `double` 로 함수를 꺼내고, 꺼낸 함수를 실행했습니다.

#### 🐍 문법 상자 · 함수를 딕셔너리에 담기 — 괄호 없이

```python
def double(args):
    return args["n"] * 2

registry = {}
registry["double"] = double
print(registry["double"]({"n": 5}))
# 10
```

| 쓰는 것 | 뜻 |
|---|---|
| `double` | 함수 **자체** — 담아 둘 수 있다 |
| `double({"n": 5})` | 함수를 **실행한 결과** — `10` |
| `registry["double"]({"n": 5})` | 꺼낸 함수를 실행한다 |

⚠ `registry["double"] = double()` 처럼 괄호를 붙이면 담는 순간 실행되어 `TypeError` 가 납니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 6-2 · 도구 두 개를 레지스트리에 등록하기</font></h3></td></tr></table>

빈 딕셔너리 `tool_registry` 를 만들고, 도구 두 개를 **이름과 함수의 짝**으로 등록하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `2` |

**💡 힌트**

1. 빈 딕셔너리는 `{}` 입니다.
2. 키는 도구 이름 문자열이고, 값은 **괄호 없는** 함수 이름입니다.
3. 키 이름은 LLM 에게 알려 준 도구 이름과 한 글자도 다르면 안 됩니다.

In [28]:
# 1. tool_registry 라는 변수를 만들어 빈 딕셔너리를 담으세요
tool_registry = {} 

# 2. tool_registry 의 "count_failed_logins" 키에 함수 count_failed_logins 를 담으세요
tool_registry["count_failed_logins"] = count_failed_logins

# 3. tool_registry 의 "lookup_ip" 키에 함수 lookup_ip 를 담으세요
tool_registry["lookup_ip"] = lookup_ip

print(len(tool_registry))

2


### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">6.2 등록되지 않은 이름은 무시하지 않습니다</mark>

5교시에 봤듯이 LLM 은 목록에 없는 일을 시키면 엉뚱한 도구를 고르거나, 없는 이름을 답할 수 있습니다.

```python
func = tool_registry.get("delete_all_logs")          # None

if func:
    result = func(args)
else:
    print("[오류] 등록되지 않은 도구: delete_all_logs")
```

1. 레지스트리에 없는 이름이면 `.get()` 이 `None` 을 반환합니다. `if func:` 로 확인합니다.
2. 없는 이름을 **무시하지 않고 알립니다.** LLM 이 도구를 지어냈거나 레지스트리를 잘못 만든 것이라, 어느 쪽이든 사람이 알아야 합니다.
3. 조용히 넘어가면 「왜 조치가 안 됐는지」 가 어디에도 남지 않습니다.

#### 🐍 문법 상자 · `.get()` — 없는 키는 `None`

```python
registry = {"count_failed_logins": 1}
print(registry.get("count_failed_logins"), registry.get("block_everything"))
# 1 None
```

| 쓰는 것 | 키가 없을 때 |
|---|---|
| `registry[name]` | `KeyError` 로 멈춘다 |
| `registry.get(name)` | `None` — `if func:` 로 확인할 수 있다 |

⚠ LLM 이 목록에 없는 도구 이름을 보낼 수 있습니다. 그래서 `[]` 가 아니라 `.get()` 으로 꺼냅니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 6-3 · route_tool_call 함수 만들기</font></h3></td></tr></table>

LLM 이 고른 `choice` 를 받아, 레지스트리에서 함수를 찾아 실행하고 결과를 반환하는 함수 `route_tool_call` 을 완성하시오. 없는 이름이면 오류를 알리고 `None` 을 반환합니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `1` 그리고 `[오류] 등록되지 않은 도구: block_everything` 그리고 `None` |

**💡 힌트**

1. 함수 안의 줄은 네 칸 들여 씁니다. 이름으로 꺼낼 때는 `tool_registry.get(name)` 입니다 (6.1).
2. 찾았으면 실행한 값을 `return` 으로 반환합니다. `return` 을 만나면 함수가 거기서 끝납니다.
3. 못 찾았을 때의 두 줄은 `if` 와 같은 깊이에 씁니다. `if` 안에 넣지 않습니다.

In [30]:
# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
# 이름 → 함수. 괄호 없이 함수 자체를 담는다
tool_registry = {"count_failed_logins": count_failed_logins, "lookup_ip": lookup_ip}
# ── 여기까지 ──


def route_tool_call(choice):                                  # 라우터 — 이름으로 함수를 찾아 실행한다
    name = choice["tool"]                                     # 고른 도구의 이름

    # 1. func 라는 변수를 만들어 tool_registry 에서 name 으로 꺼낸 함수를 담으세요
    func = tool_registry.get(name)

    # 2. func 가 있으면 (if 문)
    if func:
        # 3. func 에 choice 의 args 를 넣어 실행한 값을 return 하세요
        return func(choice["args"])

    # 4. "[오류] 등록되지 않은 도구:" 와 name 을 출력하고, None 을 return 하세요
    print("[오류] 등록되지 않은 도구:", name)
    return None


print(route_tool_call({"tool": "count_failed_logins", "args": {"user": "park.js"}}))
print(route_tool_call({"tool": "block_everything", "args": {}}))

1
[오류] 등록되지 않은 도구: block_everything
None


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 6-4 · tool_router.py 만들기</font></h3></td></tr></table>

오후의 산출물 **`tool_router.py`** 를 만드시오. 도구 둘, 레지스트리, 라우터를 파일 하나에 담습니다. 문제 6-2 와 6-3 에서 쓴 줄을 다시 씁니다.

1. 아래 셀의 번호 주석 아래에 코드를 쓰고 실행합니다. `tool_router.py` 파일이 만들어집니다.
2. 그 아래 **확인 셀**을 실행합니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | 문제 셀은 `Writing tool_router.py` (다시 실행하면 `Overwriting tool_router.py`), 확인 셀은 `1` |

**💡 힌트**

1. 레지스트리는 도구 함수를 정의한 **뒤에** 만들어야 합니다. 함수가 먼저 있어야 담을 수 있습니다.
2. 등록하는 두 줄은 문제 6-2 와, 라우터 안의 줄은 문제 6-3 과 같습니다.
3. 확인 셀에서 오류가 나면 문제 셀을 고쳐 다시 실행한 뒤, **확인 셀도 다시** 실행합니다.

In [38]:
%%writefile tool_router.py
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구
import requests                                               # 인터넷으로 요청을 보내는 도구 (9/30)


def count_failed_logins(args):                                # 도구 — 한 계정의 로그인 실패 횟수를 센다
    with open("normalized_logs.json", encoding="utf-8") as f:  # 9/29 에 정리한 로그 파일을 연다
        rows = json.load(f)                                   # 파일 내용을 리스트로 읽는다
    count = 0                                                 # 센 횟수. 0 에서 시작
    for row in rows:                                          # 로그를 한 건씩 꺼낸다
        # 계정이 같고 등급이 WARN(실패)이면
        if row["user"] == args["user"] and row["level"] == "WARN":
            count = count + 1                                 # 하나 센다
    return count                                              # 센 횟수를 돌려준다


def lookup_ip(args):                                          # 도구 — IP 의 나라와 통신사를 조회한다
    # 조회 API 에 IP 를 물어 딕셔너리로 받는다
    data = requests.get("https://ipwho.is/" + args["ip"], timeout=10).json()
    return data["country"] + " / " + data["connection"]["isp"]  # 나라와 통신사를 " / " 로 이어 돌려준다


tool_registry = {}                                            # 빈 레지스트리
# 1. tool_registry 의 "count_failed_logins" 키에 함수 count_failed_logins 를 담으세요
tool_registry["count_failed_logins"] = count_failed_logins
# 2. tool_registry 의 "lookup_ip" 키에 함수 lookup_ip 를 담으세요
tool_registry{"lookup_ip"} = lookup_ip


def route_tool_call(choice):                                  # 라우터 — 이름으로 함수를 찾아 실행한다
    name = choice["tool"]                                     # 고른 도구의 이름
    # 3. func 라는 변수를 만들어 tool_registry 에서 name 으로 꺼낸 함수를 담으세요


    if func:                                                  # 함수를 찾았으면
        return func(choice["args"])                           # 그 함수에 인자를 넣어 실행한 결과를 돌려준다

    print("[오류] 등록되지 않은 도구:", name)
    return None

Overwriting tool_router.py


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 6-5 · LLM 이 고른 도구를 라우터로 실행하기</font></h3></td></tr></table>

`llm_client.py` 와 `tool_router.py` 를 이어, 목표 하나를 처리하시오. LLM 이 도구를 고르고 라우터가 실행합니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `count_failed_logins → 1` |

**💡 힌트**

1. 고르는 줄은 5교시와 같습니다. `choose_tool(goal)` 입니다.
2. `choice` 가 `None` 일 수 있으므로 `if choice:` 로 확인한 뒤에 라우터에 넘깁니다.
3. 5교시의 `if` · `elif` 갈래가 `route_tool_call(choice)` 한 줄로 바뀌었습니다.

In [39]:
# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
goal = "park.js 계정이 로그인에 몇 번 실패했는지 알려 주세요."
# ── 여기까지 ──

# 1. choice 라는 변수에 choose_tool(goal) 의 결과를 담으세요
choice = choose_tool(goal)

# 2. choice 가 있으면 (if 문)
if choice:
    # 3. result 라는 변수에 route_tool_call(choice) 의 결과를 담으세요
    result= route_tool_call(choice)
    # 4. choice 의 tool, "→", result 를 출력하세요
    print(choice["tool"], "→", result)

count_failed_logins → 1


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 6-6 · (선택) LLM 이 도구를 고르지 못했을 때</font></h3></td></tr></table>

**선택 문제입니다.** 시간이 부족하면 건너뛰고 도전 문제로 갑니다.

도구와 상관없는 목표를 보냅니다. LLM 의 답을 읽지 못했으면 `도구를 고르지 못했습니다` 를, 읽었으면 라우터에 넘겨 결과를 출력하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `도구를 고르지 못했습니다`, 또는 `[오류] 등록되지 않은 도구: None` 과 `None`. LLM 이 문장으로 답하면 앞의 것이, `{"tool": null}` 로 답하면 뒤의 것이 나옵니다 |

**💡 힌트**

1. `choice` 가 있으면 라우터에 전달하고, 없으면 `else` 에서 문장을 출력합니다.
2. 갈래는 `if` 와 `else` 둘입니다.
3. 어느 쪽이 나와도 **프로그램은 멈추지 않고, 아무 도구도 실행되지 않았습니다.**

In [40]:
# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
goal = "오늘 점심 메뉴를 추천해 주세요."
choice = choose_tool(goal)                                    # LLM 이 고른 도구 — {"tool": 이름, "args": 인자}
# ── 여기까지 ──

# 1. choice 가 있으면 (if 문)
if choice:
    # 2. route_tool_call(choice) 의 결과를 출력하세요
    result = route_tool_call(choice)
# 3. 없으면 (else 문)

    # 4. "도구를 고르지 못했습니다" 를 출력하세요
    print("도구를 고르지 못했습니다")

<div style="background:#fff3e0; color:#4e342e; padding:12px 16px; border-radius:8px"><strong>⭐ 도전 문제 (선택)</strong><br>기본 문제를 다 푼 사람만 풉니다. 못 풀어도 괜찮습니다.</div>

#### 🐍 문법 상자 · 도구를 더해도 라우터는 그대로

```python
def count_by_ip(args):
    return 3

registry = {}
registry["count_by_ip"] = count_by_ip
print(list(registry.keys()))
# ['count_by_ip']
```

| 바꾸는 곳 | 바꾸지 않는 곳 |
|---|---|
| 도구 함수 하나 · 등록 한 줄 | `route_tool_call` |

⚠ 라우터는 이름으로 찾을 뿐이라 도구가 늘어도 고칠 필요가 없습니다. 이름만 LLM 에게 알려 준 것과 같아야 합니다.

<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 6-7 · 라우터를 고치지 않고 도구를 하나 더하기</font></h3></td></tr></table>

IP 하나에서 온 기록이 몇 건인지 세는 도구 `count_by_ip` 를 만들어 레지스트리에 등록하고, 라우터로 실행하시오. **`route_tool_call` 은 고치지 않습니다.**

| | |
|---|---|
| 🎯 나와야 하는 결과 | `5` |

**💡 힌트**

1. 도구의 모양은 `count_failed_logins` 와 같습니다. 비교하는 키가 `user` 가 아니라 `ip` 이고, 등급은 따지지 않습니다.
2. 등록은 `tool_registry["count_by_ip"] = count_by_ip` 한 줄입니다.
3. 등록한 뒤에 `route_tool_call` 에 `choice` 를 넘깁니다.

In [41]:
# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
choice = {"tool": "count_by_ip", "args": {"ip": "211.45.12.9"}}
# ── 여기까지 ──

# 1. count_by_ip 라는 함수를 만드세요. args 를 받아, normalized_logs.json 에서 ip 가 args 의 ip 와 같은 기록의 수를 return 합니다

# 2. tool_registry 의 "count_by_ip" 키에 함수 count_by_ip 를 담으세요
# 3. route_tool_call 에 choice 를 넣어 받은 결과를 출력하세요
def count_by_ip(args):
    with open("normalized_logs.json", encoding="utf-8") as f:  # 9/29 에 정리한 로그 파일을 연다
        rows = json.load(f)                                   # 파일 내용을 리스트로 읽는다
    count = 0                                                 # 센 횟수. 0 에서 시작
    for row in rows:                                          # 로그를 한 건씩 꺼낸다
        if row["ip"] == args["ip"]:
            count = count + 1                                 # 하나 센다
    return count                                              # 센 횟수를 돌려준다


tool_registry["count_by_ip"] = count_by_ip
print(route_tool_call(choice))


5


### 📋 6교시 한눈에

| 하려는 일 | 쓰는 코드 |
|---|---|
| 도구를 등록한다 | `tool_registry["lookup_ip"] = lookup_ip` (괄호 없이) |
| 이름으로 함수를 꺼낸다 | `func = tool_registry.get(name)` |
| 꺼낸 함수를 실행한다 | `func(choice["args"])` |
| 없는 이름을 알린다 | `print("[오류] 등록되지 않은 도구:", name)` |
| 한 줄로 실행한다 | `route_tool_call(choice)` |

오후 산출물 `agent_core/tool_router.py` 가 생겼습니다. 도구가 늘어도 라우터는 고치지 않습니다.

---

# <mark style="display:block; background:#ffe0b2; color:#1a1a1a; padding:6px 12px; border-radius:4px">7교시 (16:00–16:50) · 승인 게이트 — 위험한 도구 앞에 사람을 둔다</mark>

## <mark style="display:block; background:#b3e5fc; color:#0d3c61; padding:6px 12px; border-radius:4px">🔎 공부하는 법을 공부하기</mark>

설명을 읽기 전에 아래 **두 개념**을 스스로 찾아봅니다.

| 찾아볼 개념 | 알아 올 것 |
|---|---|
| **사람 개입 (human-in-the-loop)** | 자동화 흐름 가운데 어디에 사람을 두는가 |
| **되돌릴 수 없는 조치** | 보안 대응에서 자동으로 실행하면 위험한 조치에는 무엇이 있는가 |

**여기에 기록하세요** — 이 셀을 **두 번 눌러** 아래에 적습니다.

- 사람 개입 →
- 되돌릴 수 없는 조치 →

---

## <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">7 · 위험한 도구 앞에 사람을 둔다</mark>

### 왜 필요한가

1. 로그를 세거나 IP 를 조회하는 도구는 틀려도 다시 하면 됩니다.
2. **계정 잠금**은 다릅니다. 잘못 잠그면 그 사람은 일을 못 하고, 되돌리는 데 사람 손이 듭니다.
3. 5교시에 봤듯이 LLM 은 엉뚱한 도구를 고를 수 있습니다. 그래서 **되돌리기 어려운 도구 앞에는 사람의 확인**을 둡니다. 이것을 승인 게이트라고 합니다.

### 7교시에 사용하는 용어

| 용어 (읽는 법) | 뜻 | 7교시의 예 |
|---|---|---|
| 승인 게이트 (approval gate) | 위험한 실행 앞에 사람 확인을 두는 것 | `if choice["tool"] in need_approval:` |
| 사람 개입 (human-in-the-loop · 휴먼 인 더 루프) | 자동 흐름 중간에 사람이 끼는 설계 | 담당자의 답 `answer = "y"` |

아래 **준비 셀**을 먼저 실행합니다. 6교시에 만든 `tool_router.py` 를 불러오고, 도구 목록에 `lock_account` 를 더합니다.

`ModuleNotFoundError: No module named 'tool_router'` 가 나오면 6교시에 `tool_router.py` 를 만들지 못한 것입니다. 노트북 맨 아래 **「정답 6-4」 셀을 실행**한 뒤 이 셀을 다시 실행합니다.

In [ ]:
# 준비 셀 — 그대로 실행합니다. 7교시의 문제가 이 셀의 함수를 씁니다
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구
import importlib                                              # 파일을 다시 읽는 도구
import llm_client                                             # 오전에 만든 llm_client.py 를 불러온다

importlib.reload(llm_client)          # llm_client.py 를 고쳤으면 다시 읽습니다
from llm_client import call_llm, parse_llm_json               # llm_client.py 의 두 함수를 꺼내 쓴다
from tool_router import tool_registry, route_tool_call        # tool_router.py 의 레지스트리와 라우터를 꺼내 쓴다

tools_text = "쓸 수 있는 도구는 셋입니다. "                              # LLM 에게 알려 줄 도구 목록 — 이번에는 셋
tools_text = tools_text + "1) count_failed_logins: 한 계정의 로그인 실패 횟수를 센다. 인자는 user. "
tools_text = tools_text + "2) lookup_ip: IP 주소의 나라와 통신사를 조회한다. 인자는 ip. "
tools_text = tools_text + "3) lock_account: 계정을 잠근다. 인자는 user. "
tools_text = tools_text + "목표를 이루는 데 쓸 도구 하나를 골라 JSON 으로만 답하세요. "
tools_text = tools_text + '예시: {"tool": "count_failed_logins", "args": {"user": "kim01"}} '


def choose_tool(goal):                                        # 목표를 받아 LLM 이 고른 도구를 딕셔너리로 돌려주는 함수 (5.3)
    text = call_llm(tools_text + "목표: " + goal)               # 도구 목록과 목표를 이어 LLM 에 보낸다
    if text == "":                                            # 답을 받지 못했으면 (한도 초과 등)
        print("LLM 의 답을 받지 못했습니다. 1분 기다린 뒤 이 셀을 다시 실행합니다 (한도를 넘으면 429 가 옵니다)")
    return parse_llm_json(text)                               # 답을 딕셔너리로 바꿔 돌려준다


print("준비 끝 — 도구 목록에 lock_account 를 더했습니다")

### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">7.1 승인이 필요한 도구를 목록으로 정해 둡니다</mark>

도구를 **되돌릴 수 있는가**로 나눕니다. 승인이 필요한 도구의 이름을 리스트에 적어 둡니다.

```python
need_approval = ["lock_account"]

print("lock_account" in need_approval)          # True
print("lookup_ip" in need_approval)             # False
```

| 바로 실행해도 되는 도구 | 승인을 받아야 하는 도구 |
|---|---|
| `count_failed_logins` · `lookup_ip` | `lock_account` |
| 틀려도 다시 하면 된다 | 잘못하면 사람이 일을 못 한다 |

1. `in` 은 9/23 에 배웠습니다. 리스트 안에 그 값이 있으면 `True` 입니다.
2. 승인 여부를 LLM 이 정하게 하지 않습니다. **규칙은 코드에 적혀 있어야** 언제나 같게 동작합니다.
3. 오늘의 `lock_account` 는 실습이라 화면에 출력만 합니다. 실제 계정을 잠그지 않습니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 7-1 · 무엇이 보일까요</font></h3></td></tr></table>

아래 코드를 실행하면 화면에 무엇이 보일지 적어 보세요. 적은 뒤에 셀을 실행해 맞춰 봅니다.

```python
need_approval = ["lock_account", "block_ip"]

print("block_ip" in need_approval)
print("count_failed_logins" in need_approval)
```

막히면 바로 위 `7.1 승인이 필요한 도구를 목록으로 정해 둡니다` 설명을 다시 봅니다.

In [43]:
need_approval = ["lock_account", "block_ip"]

print("block_ip" in need_approval)
print("count_failed_logins" in need_approval)

True
False


✅ `True` 그리고 `False`

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 7-2 · 계정 잠금 도구를 만들고 등록하기</font></h3></td></tr></table>

계정 잠금 도구 `lock_account` 를 완성하고 레지스트리에 등록하시오. 실습이므로 화면에 출력만 합니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `[조치] 계정 잠금: guest01` 그리고 `locked` |

**💡 힌트**

1. 잠글 계정은 `args["user"]` 에 있습니다.
2. 도구는 실행 결과를 반환해야 합니다. 여기서는 문자열 `"locked"` 를 반환합니다.
3. 등록은 6교시와 같은 방법입니다. 괄호 없이 함수 이름을 담습니다.

In [46]:
def lock_account(args):                                       # 도구 — 계정을 잠근다 (실습이라 출력만 한다)
    # 1. "[조치] 계정 잠금:" 과 args 의 user 를 출력하세요
    print("[조치]" , "계정 잠금:", args["user"])

    # 2. "locked" 를 return 하세요
    return "locked"


# 3. tool_registry 의 "lock_account" 키에 함수 lock_account 를 담으세요
tool_registry["lock account"] = lock_account

print(route_tool_call({"tool": "lock_account", "args": {"user": "guest01"}}))

[오류] 등록되지 않은 도구: lock_account
None


### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">7.2 담당자의 답을 보고 실행할지 정합니다</mark>

승인이 필요한 도구는 **담당자의 답**을 확인한 뒤에 실행합니다. 실습에서는 그 답을 변수 `answer` 에 직접 적습니다.

```python
answer = "n"          # 담당자의 답

if answer == "y":
    print("실행합니다")
else:
    print("보류합니다")          # 보류합니다
```

1. `"y"` 일 때만 실행합니다. **`"y"` 가 아닌 모든 답은 실행하지 않습니다.** 잘못 적은 답이 실행으로 이어지면 안 됩니다.
2. 보류했다는 것은 문자열 `"held"` 로 남깁니다.
3. 실제 서비스에서는 담당자의 답을 승인 화면이나 메신저에서 받습니다. 5과목에서 다룹니다. 오늘은 **답을 확인하고 나서 실행한다**는 순서만 만듭니다.

#### 🐍 문법 상자 · `in` 과 `!=`

```python
need_approval = ["lock_account", "block_ip"]
tool = "lock_account"
answer = "n"
print(tool in need_approval, answer != "y")
# True True
```

| 쓰는 것 | 뜻 |
|---|---|
| `tool in need_approval` | 목록 안에 있으면 `True` (9/23) |
| `answer != "y"` | `"y"` 가 아니면 `True` — 「같지 않다」 |

⚠ `"Y"` 처럼 대문자로 답하면 `!= "y"` 가 `True` 라서 보류됩니다. 그대로 두는 것이 안전한 쪽입니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 7-3 · 승인 게이트를 함수로 만들기</font></h3></td></tr></table>

`choice` 와 담당자의 답 `answer` 를 받아, 승인이 필요한 도구인데 답이 `"y"` 가 아니면 `"held"` 를 반환하는 함수 `run_with_gate` 를 완성하시오. 그 밖에는 라우터로 실행합니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `1` · `held` · `[조치] 계정 잠금: lee.yh` · `locked` |

**💡 힌트**

1. 승인이 필요한 도구인지는 `choice["tool"] in need_approval` 로 확인합니다 (7.1).
2. `"y"` 가 아니면 그 자리에서 `return "held"` 로 함수를 끝냅니다. `!=` 는 「같지 않다」 입니다.
3. 승인이 필요 없는 도구는 답이 `"n"` 이어도 실행됩니다. 첫째 출력 `1` 이 그 경우입니다. `lock_account` 는 문제 7-2 에서 등록한 것을 씁니다.

In [51]:
# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
need_approval = ["lock_account"]                              # 사람의 승인이 필요한 도구 목록
# ── 여기까지 ──


def run_with_gate(choice, answer):                            # 승인 게이트 — 위험한 도구는 담당자의 답이 y 일 때만 실행한다
    # 1. choice 의 tool 이 need_approval 안에 있으면 (if 문)
    if choice["tool"] in need_approval:
        # 2. answer 가 "y" 가 아니면 (if 문)
        if answer != "y":
            # 3. "held" 를 return 하세요
            return "held"
    # 4. route_tool_call(choice) 의 결과를 return 하세요
    return route_tool_call(choice)



print(run_with_gate({"tool": "count_failed_logins", "args": {"user": "lee.yh"}}, "n"))
print(run_with_gate({"tool": "lock_account", "args": {"user": "lee.yh"}}, "n"))
print(run_with_gate({"tool": "lock_account", "args": {"user": "lee.yh"}}, "y"))

1
held
[오류] 등록되지 않은 도구: lock_account
None


### <mark style="display:block; background:#ffecb3; color:#1a1a1a; padding:6px 12px; border-radius:4px">7.3 무엇을 실행했는지 기록으로 남깁니다</mark>

에이전트가 한 일은 **실행했든 보류했든** 기록으로 남깁니다. 9/28 오후에 배운 `json.dump` 로 `agent_result.json` 에 저장합니다.

```python
records = []
records.append({"goal": "admin 계정이 몇 번 실패했나", "tool": "count_failed_logins", "result": 4})

with open("agent_result.json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)
```

1. 기록 한 건은 딕셔너리입니다. **목표 · 고른 도구 · 결과**를 담습니다.
2. 보류한 것도 `"result": "held"` 로 남깁니다. 남기지 않으면 「왜 조치가 안 됐는지」 를 설명할 수 없습니다.
3. `ensure_ascii=False` 는 한글을 그대로 저장하고, `indent=2` 는 두 칸 들여쓰기입니다.

#### 🐍 문법 상자 · 기록을 리스트에 모아 JSON 으로

```python
import json
records = []
records.append({"goal": "잠가 주세요", "tool": "lock_account", "result": "held"})
print(json.dumps(records, ensure_ascii=False, indent=2))
# [
#   {
#     "goal": "잠가 주세요",
#     "tool": "lock_account",
#     "result": "held"
#   }
# ]
```

| 쓰는 것 | 뜻 |
|---|---|
| `records.append({…})` | 리스트 끝에 딕셔너리 하나를 더한다 (9/22) |
| `ensure_ascii=False` | 한글을 그대로 쓴다 |
| `indent=2` | 두 칸 들여쓰기 — 사람이 읽기 좋게 |
| `json.dump(records, f, …)` | 같은 모양을 파일 `f` 에 쓴다 |

⚠ `ensure_ascii=False` 가 없으면 한글이 `\uc7a0` 같은 기호로 저장됩니다.

<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 7-4 · 실행한 일을 기록해 agent_result.json 으로 저장하기</font></h3></td></tr></table>

`choice` 를 라우터로 실행하고, **목표 · 도구 · 결과**를 딕셔너리로 만들어 `records` 에 더한 뒤, 오후의 산출물 **`agent_result.json`** 으로 저장하시오.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `[{'goal': 'jung.hw 계정이 몇 번 실패했나', 'tool': 'count_failed_logins', 'result': 1}]` — 왼쪽 파일 목록에 `agent_result.json` 이 생깁니다 |

**💡 힌트**

1. 결과는 `route_tool_call(choice)` 로 얻습니다. 딕셔너리의 키는 `goal`, `tool`, `result` 셋입니다.
2. 리스트에 더하는 것은 9/22 에 배운 `.append()` 입니다.
3. 쓰기로 열 때는 `"w"` 와 `encoding="utf-8"` 을 넣습니다. 저장은 `json.dump` 이고, 한글 그대로 두 칸 들여쓰기입니다.

In [53]:
# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
goal = "jung.hw 계정이 몇 번 실패했나"
choice = {"tool": "count_failed_logins", "args": {"user": "jung.hw"}}
records = []                                                  # 한 일을 모을 빈 리스트
# ── 여기까지 ──

# 1. result 라는 변수를 만들어 route_tool_call 에 choice 를 넣어 받은 값을 담으세요
result = route_tool_call(choice)

# 2. goal, choice 의 tool, result 를 담은 딕셔너리를 records 에 더하세요. 키는 goal, tool, result 입니다
records.append({"goal": goal, "tool": choice["tool"], "result": result})

# 3. "agent_result.json" 을 쓰기로 여세요. 변수 이름은 f 입니다 (with 문)
with open("agent_result.json", "w", encoding="utf-8") as f:
    # 4. records 를 f 에 JSON 으로 저장하세요. 한글 그대로, 두 칸 들여쓰기입니다
    json.dump(records, f, ensure_ascii=False, indent=2)

print(records)

[{'goal': 'jung.hw 계정이 몇 번 실패했나', 'tool': 'count_failed_logins', 'result': 1}]


<table width="100%" bgcolor="#A5D6A7"><tr><td bgcolor="#A5D6A7"><h3><font color="#1B5E20">✍️ 문제 7-5 · 목표 하나를 처음부터 끝까지 처리하기</font></h3></td></tr></table>

목표 하나를 **고르기 → 게이트 → 실행 → 기록**까지 한 번에 처리하시오. 담당자의 답 `answer` 는 `"y"` 입니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `[조치] 계정 잠금: admin` 그리고 `[{'goal': …, 'tool': 'lock_account', 'result': 'locked'}]` |

**💡 힌트**

1. 고르는 줄은 6교시와 같습니다. `choose_tool(goal)` 입니다.
2. 라우터 대신 **게이트 함수** `run_with_gate` 에 `choice` 와 `answer` 를 넘깁니다. 문제 7-3 에서 만든 함수입니다.
3. 기록은 문제 7-4 와 같은 딕셔너리입니다.

In [55]:
# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
goal = "admin 계정이 공격받고 있습니다. 더 이상 로그인하지 못하게 해 주세요."
answer = "y"      # 담당자의 답입니다
records = []                                                  # 한 일을 모을 빈 리스트
# ── 여기까지 ──

# 1. choice 라는 변수에 choose_tool(goal) 의 결과를 담으세요
choice = choose_tool(goal)

# 2. choice 가 있으면 (if 문)
if choice:
    # 3. result 라는 변수에 run_with_gate(choice, answer) 의 결과를 담으세요
    result = run_with_gate(choice, answer)
    # 4. goal · tool · result 를 담은 딕셔너리를 records 에 더하세요
    records.append({"goal" : goal, "tool": choice["tool"], "result": result})

print(records)

[{'goal': 'admin 계정이 공격받고 있습니다. 더 이상 로그인하지 못하게 해 주세요.', 'tool': 'count_failed_logins', 'result': 4}]


<div style="background:#fff3e0; color:#4e342e; padding:12px 16px; border-radius:8px"><strong>⭐ 도전 문제 (선택)</strong><br>기본 문제를 다 푼 사람만 풉니다. 못 풀어도 괜찮습니다.</div>

<table width="100%" bgcolor="#FFE0B2"><tr><td bgcolor="#FFE0B2"><h3><font color="#E65100">⭐ 도전 7-6 · 목표 세 개를 처리하는 작은 에이전트</font></h3></td></tr></table>

`goals` 의 목표 셋을 차례로 처리해 `agent_result.json` 에 저장하시오. 담당자의 답 `answer` 는 `"n"` 이라서 잠금은 보류됩니다.

| | |
|---|---|
| 🎯 나와야 하는 결과 | `3건 저장했습니다` — 파일을 열면 `count_failed_logins` · `lookup_ip` · `lock_account`(`held`) 세 건이 있습니다 |

**💡 힌트**

1. 반복 안의 줄은 문제 7-5 와 같습니다.
2. 반복이 끝난 뒤에 한 번만 저장합니다. 저장은 문제 7-4 와 같습니다.
3. 보류한 것도 기록에 남는지 `agent_result.json` 을 열어 확인합니다.

In [ ]:
# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
goals = [
    "admin 계정이 로그인에 몇 번 실패했는지 알려 주세요.",
    "211.45.12.9 는 어느 나라의 주소인지 알려 주세요.",
    "admin 계정이 공격받고 있습니다. 더 이상 로그인하지 못하게 해 주세요.",
]
answer = "n"      # 담당자의 답입니다
records = []                                                  # 한 일을 모을 빈 리스트
# ── 여기까지 ──

# 1. goals 를 하나씩 꺼내 반복하세요. 꺼낸 목표의 변수 이름은 goal 입니다
# 2. 반복 안에서 choice 에 choose_tool(goal) 의 결과를 담으세요
# 3. choice 가 있으면 run_with_gate(choice, answer) 로 실행하고, goal · tool · result 를 records 에 더하세요
# 4. 반복이 끝나면 records 를 "agent_result.json" 으로 저장하세요


print(f"{len(records)}건 저장했습니다")

### 📋 7교시 한눈에

| 하려는 일 | 쓰는 코드 |
|---|---|
| 승인이 필요한 도구를 정한다 | `need_approval = ["lock_account"]` |
| 승인이 필요한지 확인한다 | `if choice["tool"] in need_approval:` |
| 담당자의 답을 확인한다 | `if answer == "y":` |
| 승인하지 않으면 보류한다 | `if answer != "y":` `return "held"` |
| 한 일을 기록한다 | `records.append({"goal": …, "tool": …, "result": …})` |
| 파일로 남긴다 | `json.dump(records, f, ensure_ascii=False, indent=2)` |

판단은 LLM 이 하고, 실행은 우리 코드가 하고, 되돌리기 어려운 결정은 사람이 합니다.

---

# <mark style="display:block; background:#ffe0b2; color:#1a1a1a; padding:6px 12px; border-radius:4px">8교시 (17:00–17:30) · 오늘 만든 것 확인 · 남은 도전</mark>

새로 배우는 것은 없습니다. 오늘 만든 파일이 `agent_core` 폴더에 있는지 확인하고, 건너뛴 도전 문제를 풉니다.

아래 셀을 실행하면 오늘의 산출물 셋이 있는지 알려 줍니다.

In [ ]:
import os                                                     # 폴더 · 파일을 다루는 도구

for name in ["llm_client.py", "tool_router.py", "agent_result.json"]:
    if os.path.exists(name):
        print("있음 :", name)
    else:
        print("없음 :", name, "— 그 파일을 만드는 문제로 돌아갑니다")

### 오늘 만든 것

| 파일 | 만든 문제 | 하는 일 |
|---|---|---|
| `llm_client.py` | 오전 4-4 | LLM 에 질문을 보내고, 답을 딕셔너리로 읽는다 |
| `tool_router.py` | 오후 6-4 | 도구를 등록하고, 이름으로 찾아 실행한다 |
| `agent_result.json` | 오후 7-4 · ⭐7-6 | 에이전트가 한 일의 기록 |

### 남은 도전 문제

| 도전 | 무엇을 해 보나 |
|---|---|
| ⭐5-6 | 목표 두 개를 차례로 처리한다 |
| ⭐6-7 | 라우터를 고치지 않고 도구를 하나 더한다 |
| ⭐7-6 | 목표 세 개를 처리해 `agent_result.json` 에 남긴다 |

기본 문제를 못 끝냈으면 도전이 아니라 기본 문제부터 합니다.

### 내일 (10/7)

오늘은 로그 한 줄, 목표 하나씩 다뤘습니다. 내일은 **경보 18건을 여섯 건씩 나눠서 요약**하고, 그 결과로 일일 보고서를 만듭니다. 오늘 만든 `llm_client.py` 를 그대로 씁니다.

---

# <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">정답 · 먼저 풀어 본 뒤에 봅니다</mark>

출력 예상 문제의 답은 문제 바로 아래 ✅ 줄에 있습니다.

### 정답 5-2

In [ ]:
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구


def count_failed_logins(args):                                # 도구 — 한 계정의 로그인 실패 횟수를 센다
    with open("normalized_logs.json", encoding="utf-8") as f:  # 9/29 에 정리한 로그 파일을 연다
        rows = json.load(f)                                   # 파일 내용을 리스트로 읽는다
    count = 0                                                 # 센 횟수. 0 에서 시작
    for row in rows:                                          # 로그를 한 건씩 꺼낸다
        # 1. row 의 user 가 args 의 user 와 같고 level 이 "WARN" 이면 (if 문)
        # 계정이 같고 등급이 WARN(실패)이면
        if row["user"] == args["user"] and row["level"] == "WARN":
            # 2. count 에 1 을 더하세요
            count = count + 1                                 # 하나 센다

    # 3. count 를 return 하세요
    return count                                              # 센 횟수를 돌려준다


print(count_failed_logins({"user": "admin"}))

### 정답 5-3

In [ ]:
import importlib                                              # 파일을 다시 읽는 도구
import llm_client                                             # 오전에 만든 llm_client.py 를 불러온다

importlib.reload(llm_client)          # llm_client.py 를 고쳤으면 다시 읽습니다
from llm_client import call_llm, parse_llm_json               # llm_client.py 의 두 함수를 꺼내 쓴다

# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
tools_text = "쓸 수 있는 도구는 둘입니다. "                              # LLM 에게 알려 줄 도구 목록을 한 줄씩 이어 붙인다
tools_text = tools_text + "1) count_failed_logins: 한 계정의 로그인 실패 횟수를 센다. 인자는 user. "
tools_text = tools_text + "2) lookup_ip: IP 주소의 나라와 통신사를 조회한다. 인자는 ip. "
tools_text = tools_text + "목표를 이루는 데 쓸 도구 하나를 골라 JSON 으로만 답하세요. "
tools_text = tools_text + '예시: {"tool": "count_failed_logins", "args": {"user": "kim01"}} '
goal = "kim.cs 계정이 로그인에 몇 번 실패했는지 알려 주세요."
# ── 여기까지 ──

# 1. prompt 라는 변수를 만들어 tools_text, "목표: ", goal 을 이어 붙이세요
prompt = tools_text + "목표: " + goal                           # 도구 목록과 목표를 이어 붙인다

# 2. text 라는 변수를 만들어 call_llm 에 prompt 를 넣어 받은 답을 담고, text 를 출력하세요
text = call_llm(prompt)                                       # LLM 에 보내 답 문장을 받는다
print(text)

# 3. choice 라는 변수를 만들어 parse_llm_json 에 text 를 넣어 받은 것을 담으세요
choice = parse_llm_json(text)                                 # 답을 딕셔너리로 바꾼다. 못 읽으면 None

# 4. choice 의 tool 키와 args 키를 한 줄씩 출력하세요
print(choice["tool"])
print(choice["args"])

### 정답 5-4

In [ ]:
# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
goal = "211.45.12.9 는 어느 나라의 어느 통신사 주소인지 알려 주세요."
choice = choose_tool(goal)                                    # LLM 이 고른 도구 — {"tool": 이름, "args": 인자}
# ── 여기까지 ──

# 1. choice 의 tool 이 "count_failed_logins" 이면 (if 문)
if choice["tool"] == "count_failed_logins":
    # 2. result 라는 변수에 count_failed_logins 에 choice 의 args 를 넣어 받은 값을 담으세요
    result = count_failed_logins(choice["args"])
# 3. choice 의 tool 이 "lookup_ip" 이면 (elif 문)
elif choice["tool"] == "lookup_ip":
    # 4. result 에 lookup_ip 에 choice 의 args 를 넣어 받은 값을 담으세요
    result = lookup_ip(choice["args"])

print("실행 결과:", result)

### 정답 5-5

In [ ]:
# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
goal = "kim.cs 계정이 로그인에 몇 번 실패했는지 알려 주세요."
result = count_failed_logins({"user": "kim.cs"})      # LLM 이 고른 도구를 실행했다고 가정합니다
# ── 여기까지 ──

# 1. prompt2 라는 변수를 만들어 "목표: " + goal 뒤에 " 도구를 실행한 결과는 " 과 str(result) 를 이어 붙이세요
prompt2 = "목표: " + goal + " 도구를 실행한 결과는 " + str(result)

# 2. prompt2 뒤에 " 입니다. 이 결과로 목표에 한 문장으로 답하세요." 를 이어 붙여 다시 prompt2 에 담으세요
prompt2 = prompt2 + " 입니다. 이 결과로 목표에 한 문장으로 답하세요."

# 3. final 이라는 변수를 만들어 call_llm 에 prompt2 를 넣어 받은 답을 담으세요
final = call_llm(prompt2)                                     # 실행 결과로 한 문장 답을 받는다

print(final)

### 정답 ⭐5-6

In [ ]:
# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
goals = [
    "admin 계정이 로그인에 몇 번 실패했는지 알려 주세요.",
    "185.220.101.34 는 어느 나라의 주소인지 알려 주세요.",
]
# ── 여기까지 ──

# 1. goals 를 하나씩 꺼내 반복하세요. 꺼낸 목표의 변수 이름은 goal 입니다
# 2. 반복 안에서 choice 에 choose_tool(goal) 의 결과를 담으세요
# 3. choice 의 tool 에 따라 if · elif 로 알맞은 도구를 실행해 result 에 담으세요
# 4. choice 의 tool, "→", result 를 출력하세요
for goal in goals:
    choice = choose_tool(goal)                                # LLM 이 고른 도구 — {"tool": 이름, "args": 인자}
    if choice["tool"] == "count_failed_logins":
        result = count_failed_logins(choice["args"])
    elif choice["tool"] == "lookup_ip":
        result = lookup_ip(choice["args"])
    print(choice["tool"], "→", result)

### 정답 6-2

In [ ]:
# 1. tool_registry 라는 변수를 만들어 빈 딕셔너리를 담으세요
tool_registry = {}                                            # 빈 레지스트리

# 2. tool_registry 의 "count_failed_logins" 키에 함수 count_failed_logins 를 담으세요
tool_registry["count_failed_logins"] = count_failed_logins

# 3. tool_registry 의 "lookup_ip" 키에 함수 lookup_ip 를 담으세요
tool_registry["lookup_ip"] = lookup_ip

print(len(tool_registry))

### 정답 6-3

In [ ]:
# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
# 이름 → 함수. 괄호 없이 함수 자체를 담는다
tool_registry = {"count_failed_logins": count_failed_logins, "lookup_ip": lookup_ip}
# ── 여기까지 ──


def route_tool_call(choice):                                  # 라우터 — 이름으로 함수를 찾아 실행한다
    name = choice["tool"]                                     # 고른 도구의 이름

    # 1. func 라는 변수를 만들어 tool_registry 에서 name 으로 꺼낸 함수를 담으세요
    func = tool_registry.get(name)                            # 이름으로 함수를 꺼낸다. 없으면 None

    # 2. func 가 있으면 (if 문)
    if func:                                                  # 함수를 찾았으면
        # 3. func 에 choice 의 args 를 넣어 실행한 값을 return 하세요
        return func(choice["args"])                           # 그 함수에 인자를 넣어 실행한 결과를 돌려준다

    # 4. "[오류] 등록되지 않은 도구:" 와 name 을 출력하고, None 을 return 하세요
    print("[오류] 등록되지 않은 도구:", name)
    return None


print(route_tool_call({"tool": "count_failed_logins", "args": {"user": "park.js"}}))
print(route_tool_call({"tool": "block_everything", "args": {}}))

### 정답 6-4

In [ ]:
%%writefile tool_router.py
import json                                                   # JSON 문자열 ↔ 딕셔너리를 바꾸는 도구
import requests                                               # 인터넷으로 요청을 보내는 도구 (9/30)


def count_failed_logins(args):                                # 도구 — 한 계정의 로그인 실패 횟수를 센다
    with open("normalized_logs.json", encoding="utf-8") as f:  # 9/29 에 정리한 로그 파일을 연다
        rows = json.load(f)                                   # 파일 내용을 리스트로 읽는다
    count = 0                                                 # 센 횟수. 0 에서 시작
    for row in rows:                                          # 로그를 한 건씩 꺼낸다
        # 계정이 같고 등급이 WARN(실패)이면
        if row["user"] == args["user"] and row["level"] == "WARN":
            count = count + 1                                 # 하나 센다
    return count                                              # 센 횟수를 돌려준다


def lookup_ip(args):                                          # 도구 — IP 의 나라와 통신사를 조회한다
    # 조회 API 에 IP 를 물어 딕셔너리로 받는다
    data = requests.get("https://ipwho.is/" + args["ip"], timeout=10).json()
    return data["country"] + " / " + data["connection"]["isp"]  # 나라와 통신사를 " / " 로 이어 돌려준다


tool_registry = {}                                            # 빈 레지스트리
# 1. tool_registry 의 "count_failed_logins" 키에 함수 count_failed_logins 를 담으세요
tool_registry["count_failed_logins"] = count_failed_logins
# 2. tool_registry 의 "lookup_ip" 키에 함수 lookup_ip 를 담으세요
tool_registry["lookup_ip"] = lookup_ip


def route_tool_call(choice):                                  # 라우터 — 이름으로 함수를 찾아 실행한다
    name = choice["tool"]                                     # 고른 도구의 이름
    # 3. func 라는 변수를 만들어 tool_registry 에서 name 으로 꺼낸 함수를 담으세요
    func = tool_registry.get(name)                            # 이름으로 함수를 꺼낸다. 없으면 None

    if func:                                                  # 함수를 찾았으면
        return func(choice["args"])                           # 그 함수에 인자를 넣어 실행한 결과를 돌려준다

    print("[오류] 등록되지 않은 도구:", name)
    return None

### 정답 6-5

In [ ]:
# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
goal = "park.js 계정이 로그인에 몇 번 실패했는지 알려 주세요."
# ── 여기까지 ──

# 1. choice 라는 변수에 choose_tool(goal) 의 결과를 담으세요
choice = choose_tool(goal)                                    # LLM 이 고른 도구 — {"tool": 이름, "args": 인자}

# 2. choice 가 있으면 (if 문)
if choice:                                                    # None 이 아니면 (답을 읽었으면)
    # 3. result 라는 변수에 route_tool_call(choice) 의 결과를 담으세요
    result = route_tool_call(choice)                          # 라우터가 이름으로 찾아 실행한다
    # 4. choice 의 tool, "→", result 를 출력하세요
    print(choice["tool"], "→", result)

### 정답 6-6

In [ ]:
# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
goal = "오늘 점심 메뉴를 추천해 주세요."
choice = choose_tool(goal)                                    # LLM 이 고른 도구 — {"tool": 이름, "args": 인자}
# ── 여기까지 ──

# 1. choice 가 있으면 (if 문)
if choice:                                                    # None 이 아니면 (답을 읽었으면)
    # 2. route_tool_call(choice) 의 결과를 출력하세요
    print(route_tool_call(choice))
# 3. 없으면 (else 문)
else:
    # 4. "도구를 고르지 못했습니다" 를 출력하세요
    print("도구를 고르지 못했습니다")

### 정답 ⭐6-7

In [ ]:
# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
choice = {"tool": "count_by_ip", "args": {"ip": "211.45.12.9"}}
# ── 여기까지 ──

# 1. count_by_ip 라는 함수를 만드세요. args 를 받아, normalized_logs.json 에서 ip 가 args 의 ip 와 같은 기록의 수를 return 합니다
# 2. tool_registry 의 "count_by_ip" 키에 함수 count_by_ip 를 담으세요
# 3. route_tool_call 에 choice 를 넣어 받은 결과를 출력하세요
def count_by_ip(args):
    with open("normalized_logs.json", encoding="utf-8") as f:  # 9/29 에 정리한 로그 파일을 연다
        rows = json.load(f)                                   # 파일 내용을 리스트로 읽는다
    count = 0                                                 # 센 횟수. 0 에서 시작
    for row in rows:                                          # 로그를 한 건씩 꺼낸다
        if row["ip"] == args["ip"]:
            count = count + 1                                 # 하나 센다
    return count                                              # 센 횟수를 돌려준다


tool_registry["count_by_ip"] = count_by_ip
print(route_tool_call(choice))

### 정답 7-2

In [ ]:
def lock_account(args):                                       # 도구 — 계정을 잠근다 (실습이라 출력만 한다)
    # 1. "[조치] 계정 잠금:" 과 args 의 user 를 출력하세요
    print("[조치] 계정 잠금:", args["user"])

    # 2. "locked" 를 return 하세요
    return "locked"


# 3. tool_registry 의 "lock_account" 키에 함수 lock_account 를 담으세요
tool_registry["lock_account"] = lock_account

print(route_tool_call({"tool": "lock_account", "args": {"user": "guest01"}}))

### 정답 7-3

In [ ]:
# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
need_approval = ["lock_account"]                              # 사람의 승인이 필요한 도구 목록
# ── 여기까지 ──


def run_with_gate(choice, answer):                            # 승인 게이트 — 위험한 도구는 담당자의 답이 y 일 때만 실행한다
    # 1. choice 의 tool 이 need_approval 안에 있으면 (if 문)
    if choice["tool"] in need_approval:
        # 2. answer 가 "y" 가 아니면 (if 문)
        if answer != "y":
            # 3. "held" 를 return 하세요
            return "held"

    # 4. route_tool_call(choice) 의 결과를 return 하세요
    return route_tool_call(choice)


print(run_with_gate({"tool": "count_failed_logins", "args": {"user": "lee.yh"}}, "n"))
print(run_with_gate({"tool": "lock_account", "args": {"user": "lee.yh"}}, "n"))
print(run_with_gate({"tool": "lock_account", "args": {"user": "lee.yh"}}, "y"))

### 정답 7-4

In [ ]:
# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
goal = "jung.hw 계정이 몇 번 실패했나"
choice = {"tool": "count_failed_logins", "args": {"user": "jung.hw"}}
records = []                                                  # 한 일을 모을 빈 리스트
# ── 여기까지 ──

# 1. result 라는 변수를 만들어 route_tool_call 에 choice 를 넣어 받은 값을 담으세요
result = route_tool_call(choice)                              # 라우터가 이름으로 찾아 실행한다

# 2. goal, choice 의 tool, result 를 담은 딕셔너리를 records 에 더하세요. 키는 goal, tool, result 입니다
# 목표 · 도구 · 결과를 기록에 더한다
records.append({"goal": goal, "tool": choice["tool"], "result": result})

# 3. "agent_result.json" 을 쓰기로 여세요. 변수 이름은 f 입니다 (with 문)
with open("agent_result.json", "w", encoding="utf-8") as f:   # 결과 파일을 쓰기로 연다
    # 4. records 를 f 에 JSON 으로 저장하세요. 한글 그대로, 두 칸 들여쓰기입니다
    json.dump(records, f, ensure_ascii=False, indent=2)       # 한글 그대로, 두 칸 들여쓰기로 저장한다

print(records)

### 정답 7-5

In [ ]:
# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
goal = "admin 계정이 공격받고 있습니다. 더 이상 로그인하지 못하게 해 주세요."
answer = "y"      # 담당자의 답입니다
records = []                                                  # 한 일을 모을 빈 리스트
# ── 여기까지 ──

# 1. choice 라는 변수에 choose_tool(goal) 의 결과를 담으세요
choice = choose_tool(goal)                                    # LLM 이 고른 도구 — {"tool": 이름, "args": 인자}

# 2. choice 가 있으면 (if 문)
if choice:                                                    # None 이 아니면 (답을 읽었으면)
    # 3. result 라는 변수에 run_with_gate(choice, answer) 의 결과를 담으세요
    result = run_with_gate(choice, answer)                    # 게이트를 거쳐 실행한다. 보류면 "held"
    # 4. goal · tool · result 를 담은 딕셔너리를 records 에 더하세요
    # 목표 · 도구 · 결과를 기록에 더한다
    records.append({"goal": goal, "tool": choice["tool"], "result": result})

print(records)

### 정답 ⭐7-6

In [ ]:
# ── 미리 채워 둔 줄입니다. 고치지 않습니다 ──
goals = [
    "admin 계정이 로그인에 몇 번 실패했는지 알려 주세요.",
    "211.45.12.9 는 어느 나라의 주소인지 알려 주세요.",
    "admin 계정이 공격받고 있습니다. 더 이상 로그인하지 못하게 해 주세요.",
]
answer = "n"      # 담당자의 답입니다
records = []                                                  # 한 일을 모을 빈 리스트
# ── 여기까지 ──

# 1. goals 를 하나씩 꺼내 반복하세요. 꺼낸 목표의 변수 이름은 goal 입니다
# 2. 반복 안에서 choice 에 choose_tool(goal) 의 결과를 담으세요
# 3. choice 가 있으면 run_with_gate(choice, answer) 로 실행하고, goal · tool · result 를 records 에 더하세요
# 4. 반복이 끝나면 records 를 "agent_result.json" 으로 저장하세요
for goal in goals:
    choice = choose_tool(goal)                                # LLM 이 고른 도구 — {"tool": 이름, "args": 인자}
    if choice:                                                # None 이 아니면 (답을 읽었으면)
        result = run_with_gate(choice, answer)                # 게이트를 거쳐 실행한다. 보류면 "held"
        # 목표 · 도구 · 결과를 기록에 더한다
        records.append({"goal": goal, "tool": choice["tool"], "result": result})

with open("agent_result.json", "w", encoding="utf-8") as f:   # 결과 파일을 쓰기로 연다
    json.dump(records, f, ensure_ascii=False, indent=2)       # 한글 그대로, 두 칸 들여쓰기로 저장한다

print(f"{len(records)}건 저장했습니다")